## How to run this notebook

There are two ways. Both are safe; choose either.

**Way 1: run your own copy.** No warning, and your changes are kept.

- At the top of this page, choose **Copy to Drive** (or **File → Save a copy in Drive**).
- Your copy opens in a new tab. It is saved in your Google Drive, in the folder **Colab Notebooks**.
- In your copy, choose **Runtime → Run all**.

**Way 2: run this page directly.** Quicker, but your changes are not kept.

- Choose **Runtime → Run all**.
- Colab shows this warning:

  > Warning: This notebook was not authored by Google. This notebook is being loaded from GitHub. It may request access to your data stored with Google, or read data and credentials from other sessions. Please review the source code before executing this notebook.

- Choose **Run anyway**.

**Why it is safe.** Colab shows this warning for every notebook that does not come from Google, before it has looked at the code. This notebook does not ask for your Google account, your Drive or any other data of yours. It only saves the table it analyses, and any figures it draws, in this session, which ends when you close the page. You can read every line of the code below before you run it.

# Promotions and foot traffic: adjusted means

**Analysis of Covariance** · a worked example from the StatNex teaching guide

**The story.** A retail chain tried four promotions (coupon, buy-one-get-one, loyalty points, free gift) in 19 stores and recorded weekly sales. Busier stores sell more anyway, and the free gift happened to be tested in the busiest stores. Here the promotions are compared as if every store had the same foot traffic.

**The question.** Is the free gift really the best promotion, or did it just get the busiest stores?

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `05_anova/ancova__adjusted.csv` (19 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
Promotion,Weekly sales ($000),Foot traffic (thousands)
Coupon,37.1,8.2
Coupon,35.4,6.5
Coupon,39.1,9.0
Coupon,36.9,7.4
Coupon,34.1,5.9
Buy one get one,40.3,9.5
Buy one get one,38.0,8.8
Buy one get one,41.3,9.9
Buy one get one,37.4,8.1
Buy one get one,41.6,9.2
Loyalty points,42.0,10.6
Loyalty points,43.0,11.4
Loyalty points,40.9,10.1
Loyalty points,40.4,9.8
Free gift,44.2,12.5
Free gift,46.1,13.1
Free gift,42.7,11.8
Free gift,46.1,12.9
Free gift,47.1,14.0
""")

## Helper: eta_squared_interval

A step that no library does, written out. In the code download it is the file `eta_squared_interval.py`, which the analysis imports; here the analysis uses it directly.

In [ ]:
"""The confidence interval for partial eta squared, from the noncentral F distribution.

SciPy has the noncentral F (stats.ncf) but no function that inverts it for an interval,
so the inversion is written out (Steiger 2004). The interval is found for the
noncentrality λ and converted to partial eta squared as λ / (λ + df1 + df2 + 1).
"""

import numpy as np
from scipy import optimize, stats


def _limit(f, df1, df2, q):
    """The λ ≥ 0 under which the observed F sits at the q quantile of F(df1, df2, λ).

    0 when even λ = 0 cannot put it there: an F too small for the interval to leave 0.
    The cdf at F falls as λ rises, so the bracket opens outwards from F·df1, doubling.
    """
    def gap(lam):
        cdf = stats.f.cdf(f, df1, df2) if lam == 0 else stats.ncf.cdf(f, df1, df2, lam)
        return cdf - q

    if gap(0.0) <= 0:
        return 0.0
    start = max(f * df1, 1.0)
    step = 1.0
    while gap(max(start - step, 0.0)) <= 0:
        step *= 2
    low = max(start - step, 0.0)
    step = 1.0
    while gap(start + step) >= 0:
        step *= 2
    return optimize.brentq(gap, low, start + step, xtol=1e-14,
                           rtol=4 * np.finfo(float).eps, maxiter=500)


def partial_eta_squared_interval(f, df1, df2, level):
    """(lower, upper): the level interval for partial eta squared (Steiger 2004)."""
    total = df1 + df2 + 1
    lower = _limit(f, df1, df2, (1 + level) / 2)
    upper = _limit(f, df1, df2, (1 - level) / 2)
    return lower / (lower + total), upper / (upper + total)

## Libraries

In [ ]:
import itertools

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
from statsmodels.stats.anova import anova_lm

pd.set_option("display.width", 120)

## The data

In [ ]:
# One row per observation: the response, the factor(s) and the covariate(s). A row with a
# blank in any of these columns is left out.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
response = "Weekly sales ($000)"
factors = ["Promotion"]
covariates = ["Foot traffic (thousands)"]
data = data[[response, *factors, *covariates]].dropna()
# Each factor's levels in the order they first appear in the table, as the app lists them.
for name in factors:
    data[name] = pd.Categorical(data[name], categories=pd.unique(data[name]))

## The model

In [ ]:
# Main effects only: the covariate has one common slope in every group.
terms = [(name,) for name in [*factors, *covariates]]
# C(…, Sum) is sum-to-zero coding: each effect is measured against the average of the
# levels, which is what Type III sums of squares need. A covariate enters as it is, as a
# slope. Q("…") quotes a column name.
coded = {term: ":".join(f'C(Q({name!r}), Sum)' if name in factors else f'Q({name!r})'
                        for name in term) for term in terms}
fit = smf.ols(f'Q({response!r}) ~ ' + " + ".join(coded.values()), data=data).fit()
# Type III needs data in every combination of levels the model contains, and a covariate
# that varies within every level it is crossed with. Otherwise some effects cannot be
# estimated, and statsmodels would print numbers that mean nothing, so stop instead (the
# app then tests only what the data can estimate, and leaves blank an adjusted mean that
# cannot be estimated).
if np.linalg.matrix_rank(fit.model.exog) < fit.model.exog.shape[1]:
    raise SystemExit("An effect cannot be estimated: a combination of levels in the model has no "
                     "data, or a covariate does not vary within a level.")

## Descriptive statistics

In [ ]:
# One table per term made only of factors: each level, or each combination of levels.
# The same for the (first) covariate.
def describe(values, term):
    cells = values.groupby([data[name] for name in term], observed=True)
    table = pd.DataFrame({"n": cells.count(), "mean": cells.mean(), "SD": cells.std()})
    table["mean - SD"] = table["mean"] - table["SD"]
    table["mean + SD"] = table["mean"] + table["SD"]
    table["SE"] = table["SD"] / np.sqrt(table["n"])
    table["mean - SE"] = table["mean"] - table["SE"]
    table["mean + SE"] = table["mean"] + table["SE"]
    return table


factor_terms = [term for term in terms if all(name in factors for name in term)]
descriptive = {" * ".join(term): describe(data[response], term) for term in factor_terms}
covariate_descriptive = {" * ".join(term): describe(data[covariates[0]], term) for term in factor_terms}
for title, tables in ((response, descriptive), (covariates[0], covariate_descriptive)):
    for name, table in tables.items():
        print(f"\n{title}: {name}")
        print(table.to_string())

## Test for homogeneity of variance

In [ ]:
# Levene's test (deviations from the mean) across the cells of all the factors, on the
# model's residuals: within a group the raw response also varies with the covariate,
# which is not error.
cells = [group.to_numpy() for _, group in fit.resid.groupby([data[name] for name in factors], observed=True)]
levene = stats.levene(*cells, center="mean")
print(f"\nLevene: F = {levene.statistic:.4f}, df = ({len(cells) - 1}, {len(data) - len(cells)}), "
      f"P = {levene.pvalue:.4g}")

## Analysis of variance table

In [ ]:
# Type III sums of squares: each term is tested as if it were added last, so a factor is
# tested with the covariate held constant.
# Each F assumes independent observations, normal errors with one spread, and a
# straight-line relation to the covariate.
type3 = anova_lm(fit, typ=3).loc[list(coded.values())]
anova = pd.DataFrame(
    {"SS": type3["sum_sq"].to_numpy(), "df": type3["df"].to_numpy(),
     "MS": (type3["sum_sq"] / type3["df"]).to_numpy(), "F": type3["F"].to_numpy(),
     "P": type3["PR(>F)"].to_numpy()},
    index=[" * ".join(term) for term in terms],
)
anova.loc["Error"] = [fit.ssr, fit.df_resid, fit.ssr / fit.df_resid, np.nan, np.nan]
anova.loc["Total"] = [fit.centered_tss, len(data) - 1, np.nan, np.nan, np.nan]

## Test for the significance of regression

In [ ]:
# Do the covariates predict the response at all, once the groups are allowed for? The
# model is set against the model of the factors alone: one F for all the covariates
# together, over the error left about the regression. With one covariate it is that
# covariate's own row of the table below.
reduced = smf.ols(f'Q({response!r}) ~ ' + " + ".join(coded[(name,)] for name in factors), data=data).fit()
joint = anova_lm(reduced, fit).iloc[1]
regression = pd.DataFrame(
    [[joint["ss_diff"], joint["df_diff"], joint["ss_diff"] / joint["df_diff"], joint["F"], joint["Pr(>F)"]],
     anova.loc["Error"].to_list()],
    index=["Due to regression", "About the regression"], columns=anova.columns,
)
print()
print(regression)
print()
print(anova)

## Adjusted means

In [ ]:
# What each level's mean would be with every covariate at its overall mean, as R's
# emmeans and Stata's margins compute it. The reference grid is every combination of
# the factors' levels with each covariate at its overall mean; the fitted model predicts
# each row of it, and a level's adjusted mean averages the rows with that level. With
# one factor and one covariate this is the textbook ȳᵢ − b(x̄ᵢ − x̄): the group's mean
# moved along the regression line to where the covariate takes its overall mean, so a
# group that started with a higher covariate gives some of its advantage back.
grid = pd.DataFrame(list(itertools.product(*[data[name].cat.categories for name in factors])),
                    columns=factors)
for name in factors:
    grid[name] = pd.Categorical(grid[name], categories=data[name].cat.categories)
for name in covariates:
    grid[name] = data[name].mean()
grid["prediction"] = fit.predict(grid).to_numpy()
# Every combination of the other factors' levels counts the same (emmeans' default): the
# level's mean as if the other factors were balanced.
grid["weight"] = 1.0

rows = []
for name in factors:
    for level in data[name].cat.categories:
        group, combinations = data[data[name] == level], grid[grid[name] == level]
        raw = group[response].mean()
        adjusted_mean = np.average(combinations["prediction"], weights=combinations["weight"])
        rows.append([name, level, len(group), raw, group[covariates[0]].mean(), adjusted_mean,
                     adjusted_mean - raw])
adjusted = pd.DataFrame(rows, columns=["factor", "level", "n", "mean", "covariate mean",
                                       "adjusted mean", "adjustment"])
print()
print(adjusted.to_string(index=False))

## Effect sizes

In [ ]:
# For each term of the table, the covariate included: eta² = SS / SS total; partial eta²
# = SS / (SS + SS error), the term against the error alone; omega² = (SS - df·MS error) /
# (SS total + MS error), which removes eta²'s upward bias (Hays 1963); and partial omega²
# = (SS - df·MS error) / (SS + (N - df)·MS error) (Olejnik & Algina 2003). Partial eta²
# carries the 95% interval that inverts the noncentral F at the term's F (Steiger 2004),
# written out in eta_squared_interval.py.
# partial_eta_squared_interval: defined in the helper cell eta_squared_interval above (eta_squared_interval.py in the code download)


def size(value):
    """Cohen's (1988) reading of an eta²: 0.01 small, 0.06 medium, 0.14 large."""
    return ["negligible", "small", "medium", "large"][sum(value >= s for s in (0.01, 0.06, 0.14))]


ss_error, df_error = fit.ssr, fit.df_resid
ms_error = ss_error / df_error
ss_total, n_total = fit.centered_tss, len(data)
rows = []
for name in anova.index[:-2]:
    ss, df, f_value = anova.loc[name, ["SS", "df", "F"]]
    lower, upper = partial_eta_squared_interval(f_value, df, df_error, 0.95)
    rows += [[name, "eta²", ss / ss_total, np.nan, np.nan],
             [name, "partial eta²", ss / (ss + ss_error), lower, upper],
             [name, "omega²", (ss - df * ms_error) / (ss_total + ms_error), np.nan, np.nan],
             [name, "partial omega²", (ss - df * ms_error) / (ss + (n_total - df) * ms_error),
              np.nan, np.nan]]
effect_sizes = pd.DataFrame(rows, columns=["effect", "measure", "value", "lower", "upper"])
effect_sizes["size"] = effect_sizes["value"].map(size)
print()
print(effect_sizes.to_string(index=False))

## Normality of the residuals: the Shapiro-Wilk test

In [ ]:
# The model's residuals -- the errors it assumes normal -- rather than the responses,
# which also vary with the covariate and between the groups.
shapiro = stats.shapiro(fit.resid)
normality = [len(fit.resid), shapiro.statistic, shapiro.pvalue]
print(f"\nShapiro-Wilk on the residuals: n = {len(fit.resid)}, W = {shapiro.statistic:.4f}, "
      f"P = {shapiro.pvalue:.4g}")

## What the result says

The raw means make the free gift look best: \$45.2k of weekly sales against \$36.5k for the coupon. But the free gift stores also had the most foot traffic.

Analysis of covariance removes the effect of the covariate, foot traffic. The adjusted means are the sales each promotion would have with the same traffic. They run only from \$40.1k for the free gift to \$41.2k for buy one get one. For Promotion, F = 0.71 and P = 0.560. F compares the gaps between promotions with the noise, and the P value is the chance of gaps at least this large if the promotions did not differ.

So the free gift's lead comes from its busy stores. No difference between the promotions is shown.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.